In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('goldman_sachs.csv')

In [3]:
large_withdrawals = df[
    (df['TransactionAmount'] < 0) &
    (abs(df['TransactionAmount']) >= 50000)
]

In [5]:
large_withdrawal_count = large_withdrawals.groupby('AccountID').size().reset_index(
    name='LargeWithdrawalCount'
)

frequent_large_withdrawals = large_withdrawal_count[
    large_withdrawal_count['LargeWithdrawalCount'] >= 3
]

print(frequent_large_withdrawals)

Empty DataFrame
Columns: [AccountID, LargeWithdrawalCount]
Index: []


In [7]:
overdraft_accounts = df[df['AccountBalance'] < 0]

overdraft_summary = overdraft_accounts.groupby('AccountID').size().reset_index(
    name='OverdraftCount'
)

print(overdraft_summary)

   AccountID  OverdraftCount
0   ACC16241               1
1   ACC19178               1
2   ACC23736               1
3   ACC26973               1
4   ACC28154               1
5   ACC28292               2
6   ACC29477               1
7   ACC33287               1
8   ACC49774               1
9   ACC58667               1
10  ACC70314               1
11  ACC77533               1
12  ACC83005               1
13  ACC88449               1
14  ACC94242               1


In [9]:
risk_accounts = pd.merge(
    frequent_large_withdrawals,
    overdraft_summary,
    on='AccountID',
    how='outer'
).fillna(0)

print(risk_accounts)

    LargeWithdrawalCount AccountID  OverdraftCount
0                    0.0  ACC16241               1
1                    0.0  ACC19178               1
2                    0.0  ACC23736               1
3                    0.0  ACC26973               1
4                    0.0  ACC28154               1
5                    0.0  ACC28292               2
6                    0.0  ACC29477               1
7                    0.0  ACC33287               1
8                    0.0  ACC49774               1
9                    0.0  ACC58667               1
10                   0.0  ACC70314               1
11                   0.0  ACC77533               1
12                   0.0  ACC83005               1
13                   0.0  ACC88449               1
14                   0.0  ACC94242               1


In [11]:
balance_volatility = df.groupby('AccountID').agg(
    AvgBalance=('AccountBalance', 'mean'),
    StdBalance=('AccountBalance', 'std')
).reset_index()

In [12]:
balance_volatility['CV'] = (
    balance_volatility['StdBalance'] /
    balance_volatility['AvgBalance'].abs()
)

In [13]:
cv_threshold = balance_volatility['CV'].quantile(0.75)

high_volatility_accounts = balance_volatility[
    balance_volatility['CV'] > cv_threshold
]

print(high_volatility_accounts)

    AccountID    AvgBalance    StdBalance        CV
4    ACC11285  97401.348560  55922.732441  0.574147
5    ACC11837  84852.733695  60694.391957  0.715291
8    ACC13357  69179.806513  40614.664902  0.587088
13   ACC16241  73521.710375  45312.204045  0.616311
17   ACC18140  39960.076965  31325.244170  0.783914
25   ACC21878  67726.446770  70517.525297  1.041211
29   ACC23736  60801.533102  48785.760924  0.802377
37   ACC26026  76247.305703  45110.496180  0.591634
40   ACC26973  58738.210687  44752.439398  0.761897
41   ACC28154  63336.189234  47777.147548  0.754342
42   ACC28292  51228.003570  37077.505987  0.723774
50   ACC29477  34047.753303  53336.382019  1.566517
52   ACC30146  79029.659170  54115.060763  0.684744
55   ACC31539  45185.938342  32265.382498  0.714058
60   ACC33287  59331.981186  47466.413833  0.800014
63   ACC34568  54115.917520  35093.761551  0.648492
64   ACC34821  83955.465372  61454.350302  0.731987
68   ACC37688  63580.556277  48950.957291  0.769905
79   ACC4271

In [14]:
Q1 = df['TransactionAmount'].quantile(0.25)
Q3 = df['TransactionAmount'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


In [15]:
df['AnomalyFlag'] = (
    (df['TransactionAmount'] < lower_bound) |
    (df['TransactionAmount'] > upper_bound)
)


In [16]:
anomalies = df[df['AnomalyFlag'] == True]

print(anomalies)


     TransactionID CustomerID AccountID AccountType TransactionType  \
266             14   CUST3015  ACC21719        Loan         Deposit   

             Product    Firm Region    Manager TransactionDate  \
266  Savings Account  Firm D  North  Manager 3      22-05-2024   

     TransactionAmount  AccountBalance  RiskScore  CreditRating  TenureMonths  \
266       -30721.24789     113801.0737   0.378442           360           222   

     AnomalyFlag  
266         True  


In [18]:
anomaly_accounts = anomalies.groupby('AccountID').size().reset_index(
    name='AnomalyCount'
)

print(anomaly_accounts)

  AccountID  AnomalyCount
0  ACC21719             1


In [19]:
suspicious_customers = df[['AccountID']].drop_duplicates()


In [20]:
suspicious_customers['LargeWithdrawalFlag'] = suspicious_customers['AccountID'].isin(
    frequent_large_withdrawals['AccountID']
)


In [25]:
suspicious_customers['OverdraftFlag'] = suspicious_customers['AccountID'].isin(
    overdraft_summary['AccountID']
)


In [26]:
suspicious_customers['HighVolatilityFlag'] = suspicious_customers['AccountID'].isin(
    high_volatility_accounts['AccountID']
)


In [28]:
suspicious_customers['AnomalyFlag'] = suspicious_customers['AccountID'].isin(
    anomaly_accounts['AccountID']
)


In [29]:
suspicious_customers['SuspiciousFlag'] = (
    suspicious_customers['LargeWithdrawalFlag'] |
    suspicious_customers['OverdraftFlag'] |
    suspicious_customers['HighVolatilityFlag'] |
    suspicious_customers['AnomalyFlag']
)


In [30]:
final_suspicious_customers = suspicious_customers[
    suspicious_customers['SuspiciousFlag'] == True
]

print(final_suspicious_customers)

    AccountID  LargeWithdrawalFlag  OverdraftFlag  HighVolatilityFlag  \
4    ACC21878                False          False                True   
9    ACC28292                False           True                True   
13   ACC34821                False          False                True   
15   ACC30146                False          False                True   
21   ACC11837                False          False                True   
22   ACC88449                False           True                True   
23   ACC74631                False          False                True   
29   ACC45951                False          False                True   
31   ACC18140                False          False                True   
34   ACC49774                False           True                True   
40   ACC21719                False          False               False   
41   ACC76549                False          False                True   
42   ACC82926                False          False  